# 03 - Cepstral / feature-space EDA (Stage 2)

**Question this notebook answers:** of the controlled representation choices in the Stage-2
feat_a-feat_f ablation grid (linear vs. mel frequency warping, DCT/decorrelation on or off, coefficient
truncation, static+delta+delta-delta), which ones visibly change class separability *in
feature space*, and is any separating direction consistent across attack/generator families?

**Mode:** checks for a real ASVspoof 2019 LA download at `configs/data.yaml: dataset.root`.
If absent, builds a small **synthetic mini-manifest** (synthesised audio + fake protocol file)
so every cell below executes and is reviewable today. The active mode is printed below.

All six variants (`configs/features.yaml`) are produced by `audio_df.features.extract` --
this notebook defines no STFT/filterbank/DCT logic of its own, only calls into `features.py`,
`stats.py` and `visualization.py`, plus `sklearn.decomposition.PCA` for the exploratory
projection (explicitly *not* evidence of separability, flagged again where it's used).
Fitting **the same simple backend** to each representation (the fair-comparison step the
handoff's "EDA in feature space" section mentions) is Stage 3/T12's job (`notebook
04_simple_detection.ipynb`), not this one -- this notebook only characterises each
representation descriptively.

In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # non-interactive: this notebook is executed headless for verification
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.decomposition import PCA

from audio_df import audio, data, features, stats
from audio_df import visualization as viz
from audio_df.synthetic import build_synthetic_asvspoof_tree


def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(6):
        if (cur / "pyproject.toml").exists():
            return cur
        cur = cur.parent
    raise RuntimeError(f"could not find repo root (pyproject.toml) from {start}")


REPO_ROOT = _find_repo_root(Path.cwd())
pd.set_option("display.width", 120)

In [ ]:
_data_cfg_path = REPO_ROOT / "configs" / "data.yaml"
_real_cfg = yaml.safe_load(_data_cfg_path.read_text())
_real_root = Path(_real_cfg["dataset"]["root"])
if not _real_root.is_absolute():
    _real_root = REPO_ROOT / _real_root

USE_REAL_DATA = _real_root.exists() and any(_real_root.rglob("*.flac"))

if USE_REAL_DATA:
    MODE = "REAL DATA"
    cfg = _real_cfg
    cfg["dataset"]["root"] = str(_real_root)
    cfg["dataset"]["protocol_dir"] = str(REPO_ROOT / cfg["dataset"]["protocol_dir"])
    _tmpdir_handle = None
else:
    MODE = "SYNTHETIC MINI-MANIFEST (no ASVspoof download found)"
    _tmpdir_handle = tempfile.TemporaryDirectory(prefix="asvspoof_synth_")
    cfg = build_synthetic_asvspoof_tree(Path(_tmpdir_handle.name), seed=777)

print(f"MODE: {MODE}")

manifest = data.build_manifest(cfg)
print(f"manifest: {len(manifest)} rows")

FEAT_CFG = yaml.safe_load((REPO_ROOT / "configs" / "features.yaml").read_text())
FEAT_CFG.setdefault("sr", 16000)
VARIANTS = list(FEAT_CFG["variants"].keys())
print(f"variants: {VARIANTS}")

In [ ]:
# Keep this fast in both modes: a small stratified sample, six variants extracted per
# utterance (mean-pooled over frames into one vector per utterance -- the same aggregation
# convention notebook 02 uses for spectra, so per-utterance rows can feed distributions,
# effect sizes, correlation and PCA without a variable-length-sequence complication).
N_PER_CELL = 15
rng = np.random.default_rng(1)


def _stratified_sample(df: pd.DataFrame, n_per_cell: int) -> pd.DataFrame:
    parts = []
    for _, group in df.groupby(["split", "label"], dropna=False):
        take = min(n_per_cell, len(group))
        idx = rng.choice(group.index.to_numpy(), size=take, replace=False)
        parts.append(df.loc[idx])
    return pd.concat(parts).reset_index(drop=True)


eda_df = _stratified_sample(manifest, N_PER_CELL)
print(f"EDA sample: {len(eda_df)} of {len(manifest)} manifest rows")

variant_vecs: dict[str, np.ndarray] = {}
meta_rows = []
for variant in VARIANTS:
    vecs = []
    for row in eda_df.itertuples(index=False):
        x, sr = audio.load_audio(row.path, target_sr=FEAT_CFG["sr"])
        feat = features.extract(x, variant, FEAT_CFG)  # [T, D]
        vecs.append(feat.mean(axis=0))
    variant_vecs[variant] = np.stack(vecs)
    if not meta_rows:
        meta_rows = list(eda_df[["utt_id", "label", "attack_id", "split"]].itertuples(index=False))

meta_df = pd.DataFrame(meta_rows, columns=["utt_id", "label", "attack_id", "split"])
for v, mat in variant_vecs.items():
    print(f"{v}: D={mat.shape[1]}")

## Per-dimension distributions (one variant)

`feat_b_lfcc_full` (linear filterbank, full DCT) -- the first four coefficients, pooled by class.

In [ ]:
demo_variant = "feat_b_lfcc_full"
demo_mat = variant_vecs[demo_variant]
labels = meta_df["label"].to_numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for dim, ax in enumerate(axes):
    values_by_class = {lab: demo_mat[labels == lab, dim] for lab in np.unique(labels)}
    viz.plot_class_distributions(values_by_class, ax=ax)
    ax.set_title(f"{demo_variant} coeff[{dim}]")
fig.tight_layout()
plt.show()

### Observation
Each panel is the pooled bonafide/spoof distribution of one DCT coefficient (or filterbank
channel, for a non-DCT variant) of one representation.

### Inference
As with Stage 1 scalar features, a visibly offset panel is a candidate separating dimension
for *this* representation, not a claim that this coefficient index means the same thing
across variants (frequency warping and truncation change what each index represents).

## Effect size per dimension, across all six variants

Same `stats.per_bin_effect_size` used in notebook 02, here applied along the coefficient axis
instead of the frequency-bin axis -- it is a standardized-mean-difference computation and is
agnostic to what the columns represent.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, variant in zip(axes.ravel(), VARIANTS, strict=False):
    mat = variant_vecs[variant]
    eff = stats.per_bin_effect_size(mat[labels == "spoof"], mat[labels == "bonafide"])
    viz.plot_effect_size_per_bin(eff, np.arange(len(eff)), ax=ax)
    ax.set_xlabel("coefficient index")
    ax.set_title(f"{variant} (D={mat.shape[1]})")
fig.tight_layout()
plt.show()

### Observation
Each panel is the per-dimension standardized effect size (spoof - bonafide) for one variant,
on the same y-scale question (not the same y-axis limits) so the *pattern* across dimensions
can be compared.

### Inference
This is a descriptive comparison of where each representation concentrates its class
difference (e.g. low-index vs. high-index coefficients), not a claim about which
representation is "better" -- ranking representations for detection requires the same fixed
backend fit to each (Stage 3/T12), which this notebook does not do.

## Correlation matrices: does the DCT decorrelate, as claimed?

`feat_a_linear_logfb` (no DCT) vs. `feat_b_lfcc_full` (same filterbank, full orthonormal DCT) on
bonafide utterances only, so any difference in off-diagonal structure is attributable to the
DCT step alone, not a class or attack effect.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, variant in zip([ax1, ax2], ["feat_a_linear_logfb", "feat_b_lfcc_full"], strict=False):
    mat = variant_vecs[variant][labels == "bonafide"]
    corr = np.corrcoef(mat.T)
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_title(f"{variant} (bonafide only)")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
plt.show()

for variant in ["feat_a_linear_logfb", "feat_b_lfcc_full"]:
    mat = variant_vecs[variant][labels == "bonafide"]
    corr = np.corrcoef(mat.T)
    off_diag = corr[~np.eye(corr.shape[0], dtype=bool)]
    print(f"{variant}: mean |off-diagonal correlation| = {np.mean(np.abs(off_diag)):.3f}")

### Observation
The printed mean absolute off-diagonal correlation summarises each heatmap into one number.

### Inference
If `feat_b_lfcc_full`'s value is markedly lower than `feat_a_linear_logfb`'s, that is consistent with
the DCT's decorrelating role in the pipeline (per `features.py`'s module docstring and
`research/knowledge/methods/lfcc-mfcc-dct.md`: the orthonormal DCT-II is the transform whose
matrix satisfies `O @ O.T = I`, which is the algebraic property behind decorrelation of a
signal with roughly DCT-basis-aligned covariance). This is descriptive of the pooled
mean-pooled-over-time vectors used here (a coarser summary than a full per-frame covariance),
not a formal proof for this particular sample.

## PCA (exploratory visualization only -- not evidence of separability)

Per the handoff: PCA/t-SNE/UMAP are exploratory pictures, never evidence of separability on
their own. Shown here for the same two variants as the correlation comparison, plus the
truncated/mel variants, to see whether an obvious 2D linear-projection gap appears at all --
its *absence* is not evidence of inseparability either, since PCA is not distance- or
margin-optimal for classification.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, variant in zip(axes.ravel(), VARIANTS, strict=False):
    mat = variant_vecs[variant]
    proj = PCA(n_components=2, random_state=0).fit_transform(mat)
    marker_by_class = {"bonafide": "o", "spoof": "s"}
    for lab in np.unique(labels):
        mask = labels == lab
        color = viz.CLASS_COLORS.get(lab, "#333333")
        marker = marker_by_class.get(lab, "x")
        ax.scatter(
            proj[mask, 0], proj[mask, 1],
            color=color, marker=marker, s=18, alpha=0.7, label=lab,
        )
    ax.set_title(variant)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

### Observation
Each panel is a 2-component PCA projection of one representation's per-utterance vectors,
coloured by class.

### Inference
A visible cluster gap here is suggestive but **not proof** of separability (PCA maximises
variance, not class separation, and 2 components can hide a separation living in a different
direction). Absence of a visible gap likewise does not prove inseparability. This plot is a
sanity check alongside the effect-size table above, never a substitute for it.

## Attack-conditioned view

Same coefficient as the per-dimension distribution above, now split by attack/generator
instead of pooled class.

In [ ]:
attack_ids = meta_df["attack_id"].to_numpy()
fig, ax = plt.subplots(figsize=(8, 4.5))
values_by_attack = {
    a: demo_mat[attack_ids == a, 0]
    for a in sorted(np.unique(attack_ids))
    if (attack_ids == a).sum() >= 2
}
viz.plot_class_distributions_by_attack(values_by_attack, ax=ax)
ax.set_title(f"{demo_variant} coeff[0], by attack/generator")
plt.show()

### Observation
Distribution of the first coefficient of `feat_b_lfcc_full`, one curve per attack family
(bonafide is `'-'`).

### Inference
As in notebook 02, watch for attacks on opposite sides of the bonafide curve -- exactly the
generator-conditioned reversal the handoff warns pooled class plots can hide. Any cue that
holds for some attacks and reverses for others belongs to those generators specifically, not
to "synthetic speech" as a category.

## Temporal trajectories

Coefficient 0 of `feat_b_lfcc_full` across time, averaged within class, after cropping every
utterance to the project's standard 4-second window (`configs/data.yaml: audio.crop_seconds`)
so utterances align on a shared time axis.

In [ ]:
crop_seconds = _real_cfg.get("audio", {}).get("crop_seconds", 4.0)
n_samples = int(crop_seconds * FEAT_CFG["sr"])

traj_by_class: dict[str, list[np.ndarray]] = {"bonafide": [], "spoof": []}
for row in eda_df.itertuples(index=False):
    x, sr = audio.load_audio(row.path, target_sr=FEAT_CFG["sr"])
    x = audio.fix_length(x, n_samples, mode="center")
    feat = features.extract(x, demo_variant, FEAT_CFG)  # [T, D], T now identical across rows
    traj_by_class[row.label].append(feat[:, 0])

traj_mean = {lab: np.mean(np.stack(v), axis=0) for lab, v in traj_by_class.items()}
fig, ax = plt.subplots(figsize=(8, 4))
viz.plot_feature_trajectory(traj_mean, ax=ax)
ax.set_title(f"{demo_variant} coeff[0], mean trajectory over a {crop_seconds:.0f}s crop")
plt.show()

### Observation
Each curve is the class-mean trajectory of one coefficient across a fixed-length crop's
frames (not a single utterance).

### Inference
A consistent gap across most of the time axis suggests a temporally stable cue rather than an
edge/onset artifact of cropping; a gap concentrated near the boundaries would instead point at
the cropping/padding procedure itself (worth checking against the Stage 0 silence-leakage
audit in notebook 01 before trusting it as an acoustic cue).

In [ ]:
print(f"MODE was: {MODE}")

## Summary

This notebook characterised the six Stage-2 representations (`configs/features.yaml`)
descriptively: per-dimension distributions, per-dimension effect size, correlation structure,
an exploratory PCA projection, and attack-conditioned/temporal views. None of this ranks the
representations for detection -- that comparison requires fitting the *same* simple backend to
each (Stage 3, `notebook 04_simple_detection.ipynb`, owned by T12), holding the split, seed and
backend fixed so any difference is attributable to the representation alone.